# Visualize Best Agent from Checkpoint

This notebook loads the best agent from a checkpoint file, displays its program code, and visualizes it playing FlappyBird with human rendering.


In [1]:
# Import required libraries
import flappy_bird_env  # noqa
import numpy as np
import os
from pathlib import Path
from scipy.special import expit

# Disable headless mode for human rendering
os.environ.pop('SDL_VIDEODRIVER', None)

try:
    import pygame
    pygame.init()
    print("✓ Pygame initialized for human rendering")
except Exception as e:
    print(f"Warning: Could not initialize pygame: {e}")

import gymnasium as gym
from evolution_engine import EvolutionEngine
from memory_system import MemoryConfig, MemoryType
from evaluator import FlappyBirdEvaluator, FlappyBirdEvaluatorConfig


✓ Pygame initialized for human rendering


## Load Checkpoint

Load the checkpoint file containing the saved population and best agent.


In [2]:
# Path to checkpoint file (adjust if needed)
checkpoint_path = "checkpoints/best_population.pkl"

# Check if file exists
if not Path(checkpoint_path).exists():
    print(f"❌ Checkpoint file not found: {checkpoint_path}")
    print("Please update checkpoint_path to point to your checkpoint file.")
else:
    print(f"Loading checkpoint from: {checkpoint_path}")
    checkpoint = EvolutionEngine.load_checkpoint(checkpoint_path)
    
    population = checkpoint['population']
    generation = checkpoint['generation']
    fitness = checkpoint['fitness']
    config = checkpoint.get('config', None)
    
    print(f"✓ Checkpoint loaded successfully!")
    print(f"  Generation: {generation}")
    print(f"  Best fitness: {fitness:.4f}")
    print(f"  Population size: {len(population)}")
    
    # Get the best agent
    best_agent = population.get_best()
    if population.best_ever is not None:
        best_agent = population.best_ever
        print(f"  Best ever generation: {population.best_ever_generation}")
    
    print(f"  Best agent ID: {best_agent.id}")
    print(f"  Best agent fitness: {best_agent.fitness:.4f}")
    print(f"  Program length: {len(best_agent.program)}")


Loading checkpoint from: checkpoints/best_population.pkl
✓ Checkpoint loaded successfully!
  Generation: 263
  Best fitness: 0.6940
  Population size: 500
  Best ever generation: 263
  Best agent ID: 79127
  Best agent fitness: 0.6940
  Program length: 12


## Display Program Code

Show the full program and the effective program (with introns removed).


In [3]:
# Get memory config from population
memory_cfg = population.memory_config

# Get output registers (needed for effective program calculation)
# We'll create a temporary evaluator to get the output_registers
# Or we can infer from the checkpoint config if available
output_registers = [(MemoryType.SCALAR, 0)]  # Default, adjust if your config uses different

# Try to get from checkpoint config if available
if config and hasattr(config, 'evaluator'):
    # If evaluator config is saved, we could extract from there
    pass

print("="*80)
print("FULL PROGRAM")
print("="*80)
print()

for i, instr in enumerate(best_agent.program.instructions):
    print(f"{i:4d}: {instr.to_resolved_str(memory_cfg)}")

print()
print("="*80)
print(f"Total: {len(best_agent.program)} instructions")
print("="*80)


FULL PROGRAM

   0: vector[6→6] = automl_vector_max(vector[4878→6], vector[7129→1])
   1: vector[5→5] = automl_scalar_vector_mul(obs_scalar[7471→211], vector[5190→6])
   2: matrix[4→4] = automl_matrix_element_constant(matrix[7085→5], scalar[7929→1], scalar[2921→1], obs_scalar[8346→118])
   3: matrix[0→0] = cv_gaussian_blur(obs_matrix[2441→full_matrix0], obs_scalar[2897→477], obs_scalar[9695→15])
   4: matrix[0→0] = automl_matrix_uniform(scalar[653→5], scalar[8218→2])
   5: vector[1→1] = automl_matrix_norm_axis1(matrix[6623→7])
   6: matrix[1→1] = automl_matrix_div(matrix[9404→4], matrix[9239→7])
   7: scalar[0→0] = automl_scalar_gaussian(scalar[1868→4], scalar[1509→5])
   8: matrix[2→2] = automl_matrix_sub(matrix[4723→3], obs_matrix[1720→full_matrix0])
   9: vector[0→0] = automl_matrix_norm_axis1(matrix[6553→1])
  10: vector[2→2] = automl_vector_heaviside(obs_vector[483→col21])
  11: vector[1→1] = automl_scalar_broadcast(scalar[7601→1])

Total: 12 instructions


In [4]:
# Show effective program (with introns removed)
effective_program = best_agent.get_effective_program(output_registers)

print("="*80)
print("EFFECTIVE PROGRAM (INTRONS REMOVED)")
print("="*80)
print()

if len(effective_program.instructions) > 0:
    for i, instr in enumerate(effective_program.instructions):
        print(f"{i:4d}: {instr.to_resolved_str(memory_cfg)}")
else:
    print("  (No effective instructions found)")

print()
print("="*80)
print(f"Effective: {len(effective_program)} instructions (out of {len(best_agent.program)} total)")
if len(best_agent.program) > 0:
    effective_ratio = len(effective_program) / len(best_agent.program)
    print(f"Effective code rate: {effective_ratio:.3f} ({effective_ratio*100:.1f}%)")
print("="*80)


EFFECTIVE PROGRAM (INTRONS REMOVED)

   0: scalar[0→0] = automl_scalar_gaussian(scalar[1868→4], scalar[1509→5])

Effective: 1 instructions (out of 12 total)
Effective code rate: 0.083 (8.3%)


## Create Evaluator with Human Rendering

Set up the FlappyBird evaluator with human rendering mode to visualize the agent playing.


In [5]:
# Create evaluator config for visualization
# Use the same config as training, but with human rendering
evaluator_config = FlappyBirdEvaluatorConfig(
    env_id="FlappyBird-v0",
    episodes=3,  # Run 3 episodes to see the agent play
    max_steps=500,
    output_register=0,
    render_mode="human",  # Human rendering to see the game
    rng_seed=42,
    patch_strategy="quantized",
    color_channel=2,  # Blue channel (adjust if your training used different)
    normalize=True,
    quantization_factor=0.03,  # Match your training config
    output_registers=[(MemoryType.SCALAR, 0)],
    n_jobs=1,  # Sequential for visualization
)

print("Creating FlappyBird evaluator with human rendering...")
evaluator = FlappyBirdEvaluator(config=evaluator_config)
print("✓ Evaluator created!")
print(f"  Episodes: {evaluator.episodes}")
print(f"  Max steps per episode: {evaluator.max_steps}")
print(f"  Render mode: {evaluator.config.render_mode}")
print()
print("⚠️  NOTE: FlappyBird windows will appear when you run the next cell!")
print("   Close the windows or press Ctrl+C to stop.")


Creating FlappyBird evaluator with human rendering...
✓ Evaluator created!
  Episodes: 3
  Max steps per episode: 500
  Render mode: human

⚠️  NOTE: FlappyBird windows will appear when you run the next cell!
   Close the windows or press Ctrl+C to stop.


## Visualize Agent Playing

Run the agent and watch it play FlappyBird. The game windows will appear showing the agent's performance.


In [6]:
# Run the agent and visualize
print("="*80)
print("RUNNING BEST AGENT")
print("="*80)
print()

total_reward = 0.0
episode_rewards = []

for episode_idx in range(evaluator.episodes):
    print(f"\nEpisode {episode_idx + 1}/{evaluator.episodes}")
    print("-" * 80)
    
    # Seed the episode
    episode_seed = int((evaluator.config.rng_seed + episode_idx * 100) % (2**31))
    observation, _ = evaluator.env.reset(seed=episode_seed)
    observation = np.asarray(observation, dtype=np.float32)
    
    # Copy memory for this episode
    memory = best_agent.memory.copy()
    episode_reward = 0.0
    steps = 0
    
    for step in range(evaluator.max_steps):
        # Process observation
        processed_observations, obs_type = evaluator._process_observation(observation)
        
        # Load observations into memory
        if obs_type == 'vector':
            memory.load_observation({'vector': processed_observations})
        else:  # obs_type == 'matrix'
            memory.load_observation({'matrix': processed_observations})
        
        # Execute the program
        best_agent.get_effective_program(evaluator.output_registers).execute(memory)
        
        # Read action from output register
        action_value = memory.read_scalar(evaluator.output_register)
        normalized = expit(action_value)  # Sigmoid
        action = 1 if normalized >= 0.5 else 0
        
        # Take step in environment
        observation, reward, terminated, truncated, _ = evaluator.env.step(action)
        observation = np.asarray(observation, dtype=np.float32)
        episode_reward += reward
        steps += 1
        
        if terminated or truncated:
            break
    
    episode_rewards.append(episode_reward)
    total_reward += episode_reward
    
    print(f"  Steps: {steps}")
    print(f"  Reward: {episode_reward:.2f}")
    print(f"  Action value (scalar[0]): {action_value:.4f}")
    print(f"  Normalized (sigmoid): {normalized:.4f}")
    print(f"  Action chosen: {'FLAP' if action == 1 else 'NOOP'}")

print()
print("="*80)
print("SUMMARY")
print("="*80)
print(f"Total episodes: {evaluator.episodes}")
print(f"Average reward: {total_reward / evaluator.episodes:.2f}")
print(f"Rewards per episode: {[f'{r:.2f}' for r in episode_rewards]}")
print("="*80)

# Close the evaluator
evaluator.close()
print("\n✓ Visualization complete!")


RUNNING BEST AGENT


Episode 1/3
--------------------------------------------------------------------------------
  Steps: 21
  Reward: 0.02
  Action value (scalar[0]): -0.5120
  Normalized (sigmoid): 0.3747
  Action chosen: NOOP

Episode 2/3
--------------------------------------------------------------------------------
  Steps: 83
  Reward: 0.08
  Action value (scalar[0]): -0.7993
  Normalized (sigmoid): 0.3102
  Action chosen: NOOP

Episode 3/3
--------------------------------------------------------------------------------
  Steps: 83
  Reward: 0.08
  Action value (scalar[0]): -1.1682
  Normalized (sigmoid): 0.2372
  Action chosen: NOOP

SUMMARY
Total episodes: 3
Average reward: 0.06
Rewards per episode: ['0.02', '0.08', '0.08']

✓ Visualization complete!


## Additional Information

Display additional details about the best agent's memory and constants.


In [7]:
# Display memory information
print("="*80)
print("BEST AGENT MEMORY INFORMATION")
print("="*80)
print()

memory = best_agent.memory

print("Scalar registers (working):")
for i in range(min(8, memory.n_scalar)):
    print(f"  scalar[{i}]: {memory.scalars[i]:.6f}")

print()
print("Vector registers (working):")
for i in range(min(3, memory.n_vector)):
    vec_str = ", ".join([f"{v:.3f}" for v in memory.vectors[i][:5]])
    if len(memory.vectors[i]) > 5:
        vec_str += "..."
    print(f"  vector[{i}]: [{vec_str}]")

print()
print("Matrix registers (working):")
for i in range(min(3, memory.n_matrix)):
    print(f"  matrix[{i}]: shape {memory.matrices[i].shape}, "
          f"mean={memory.matrices[i].mean():.4f}, "
          f"std={memory.matrices[i].std():.4f}")

print()
print("Observation registers:")
if memory.n_obs_matrix > 0:
    print(f"  obs_matrix[0]: shape {memory.obs_matrices[0].shape}")

print("="*80)


BEST AGENT MEMORY INFORMATION

Scalar registers (working):
  scalar[0]: 0.197356
  scalar[1]: -0.234868
  scalar[2]: -0.439534
  scalar[3]: -0.320100
  scalar[4]: -1.283521
  scalar[5]: -0.955726
  scalar[6]: 1.461748
  scalar[7]: -1.580413

Vector registers (working):
  vector[0]: [0.484, 0.567, 0.379, -1.299, 0.909...]
  vector[1]: [0.079, -0.225, -0.710, 0.122, 0.854...]
  vector[2]: [0.459, -1.154, -0.343, -0.538, 0.445...]

Matrix registers (working):
  matrix[0]: shape (22, 22), mean=-0.0314, std=0.3737
  matrix[1]: shape (22, 22), mean=0.0156, std=0.3863
  matrix[2]: shape (22, 22), mean=0.0003, std=0.4039

Observation registers:
  obs_matrix[0]: shape (22, 22)
